# 第 501 个分子：Train / Validation / Test 小实验

> 目标不是训练一个能发表的溶解度模型，而是**亲手看见训练循环与数据分工**。

这个 notebook 使用固定随机种子生成 500 条 **ESOL 风格的合成 descriptor 数据**。它借用了 `logP / MW / aromatic proportion / rotatable bonds → logS` 的任务外形，但所有数值均为教学合成，**不是 Delaney ESOL 数据，也不能用于化学结论**。

游戏规则：

1. Train 用来更新 parameters。
2. Validation 用来选择 learning rate。
3. Test 先封存；方案冻结后只打开一次。
4. 最后故意做一次 leakage，看看为什么“分数漂亮”不等于方法正确。

只依赖 NumPy 与 Matplotlib；在本地 Jupyter、VS Code Notebook 或 Google Colab 均可运行。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260820
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.25})
print(f"Random seed fixed at {SEED}. Test is still sealed. [SEALED]")

## 1. 生成 500 个“教学分子”

四列输入分别模拟 calculated logP、molecular weight、aromatic heavy-atom proportion 和 rotatable-bond count。合成 `logS` 同时含线性项、一个轻微非线性项和测量噪声，所以线性模型能学到东西，但不会完美。

In [ ]:
n = 500
logp = rng.normal(2.2, 1.1, n)
mw = np.clip(rng.normal(310, 85, n), 90, 650)
aromatic_fraction = rng.beta(2.2, 3.0, n)
rotatable_bonds = rng.poisson(4.0, n).astype(float)
X = np.column_stack([logp, mw, aromatic_fraction, rotatable_bonds])
noise = rng.normal(0, 0.38, n)
y = (-0.65 * logp - 0.0032 * mw - 0.8 * aromatic_fraction
     + 0.06 * rotatable_bonds + 0.12 * np.sin(logp * 2.0) + noise)
feature_names = ["calc_logP", "MW", "aromatic_fraction", "rotatable_bonds"]
print("X shape:", X.shape, "| y shape:", y.shape)
print("Reminder: these are synthetic teaching values, not real ESOL measurements.")

## 2. 先 split，再 fit 任何预处理

这里用 350 / 75 / 75 只是为了对应课堂图。比例不是标准答案。代码把 Test 放入 `sealed_test`，提醒我们不要在模型选择期间读取它。

In [ ]:
order = rng.permutation(n)
train_idx, val_idx, test_idx = order[:350], order[350:425], order[425:]
X_train_raw, y_train = X[train_idx], y[train_idx]
X_val_raw, y_val = X[val_idx], y[val_idx]
sealed_test = {"X_raw": X[test_idx], "y": y[test_idx]}  # do not inspect yet

# Preprocessing learns state, so fit it on Train only.
mu = X_train_raw.mean(axis=0)
sigma = X_train_raw.std(axis=0)
sigma[sigma == 0] = 1.0
X_train = (X_train_raw - mu) / sigma
X_val = (X_val_raw - mu) / sigma

print("Train:", len(train_idx), "| Validation:", len(val_idx), "| Test: 75 [SEALED]")
print("Scaler statistics came from Train only. [OK]")

## 3. 从零写一个 minibatch 训练循环

为了让每个角色都看得见，这里不用机器学习框架。模型是 `ŷ = Xw + b`，loss 是 MSE，optimizer 是最基础的 gradient descent。

In [ ]:
def mse(y_true, y_pred):
    return float(np.mean((y_pred - y_true) ** 2))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_pred - y_true)))

def train_linear(Xtr, ytr, Xva, yva, learning_rate=0.03, epochs=120, batch_size=50, seed=SEED):
    local_rng = np.random.default_rng(seed)
    w = np.zeros(Xtr.shape[1])
    b = 0.0
    history = {"train": [], "val": []}

    for _ in range(epochs):
        for idx in np.array_split(local_rng.permutation(len(Xtr)),
                                  int(np.ceil(len(Xtr) / batch_size))):
            xb, yb = Xtr[idx], ytr[idx]                 # ① batch
            pred = xb @ w + b                          # ② prediction
            error = pred - yb                          # ③ compare with y
            grad_w = 2.0 * xb.T @ error / len(idx)     # loss gradient
            grad_b = 2.0 * error.mean()
            w -= learning_rate * grad_w                # ④ update θ
            b -= learning_rate * grad_b
        history["train"].append(mse(ytr, Xtr @ w + b))
        history["val"].append(mse(yva, Xva @ w + b))
    return {"w": w, "b": b, "history": history, "learning_rate": learning_rate}

print("Training function ready: batch -> prediction -> loss signal -> update -> repeat.")

## 4. 只看 Validation 选择 learning rate

先运行三个候选方案。不要读取 Test。观察：太小的步长可能在固定 epochs 内走得慢；过大的步长可能不稳定。具体边界依赖数据与实现，不要把某个数字背成通用答案。

In [ ]:
candidates = [
    train_linear(X_train, y_train, X_val, y_val, learning_rate=0.001),
    train_linear(X_train, y_train, X_val, y_val, learning_rate=0.03),
    train_linear(X_train, y_train, X_val, y_val, learning_rate=0.45),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, run in zip(axes, candidates):
    ax.plot(run["history"]["train"], label="Train")
    ax.plot(run["history"]["val"], label="Validation")
    ax.set_title(f"learning rate = {run['learning_rate']}")
    ax.set_xlabel("epoch")
axes[0].set_ylabel("MSE")
axes[-1].legend()
plt.tight_layout()

for run in candidates:
    print(f"lr={run['learning_rate']:<6} final train={run['history']['train'][-1]:.4f} "
          f"final val={run['history']['val'][-1]:.4f}")
best = min(candidates, key=lambda run: run["history"]["val"][-1])
print(f"\nChosen by Validation only: learning rate = {best['learning_rate']} [OK]")
print("Test is still sealed. [SEALED]")

## 5. 冻结方案，然后只打开一次 Test

到这里，representation、scaler、model、learning rate、epochs 与 batch size 都已经固定。现在才读取 Test，并且只做 `transform`，不重新 `fit` scaler。

In [ ]:
X_test = (sealed_test["X_raw"] - mu) / sigma  # transform with Train statistics
y_test = sealed_test["y"]
test_pred = X_test @ best["w"] + best["b"]
print("[OPEN] Test opened after the pipeline was frozen.")
print(f"Test MAE  = {mae(y_test, test_pred):.4f} log mol/L")
print(f"Test RMSE = {np.sqrt(mse(y_test, test_pred)):.4f} log mol/L")

plt.scatter(y_test, test_pred, alpha=0.75, edgecolor="white")
lo = min(y_test.min(), test_pred.min())
hi = max(y_test.max(), test_pred.max())
plt.plot([lo, hi], [lo, hi], "--", color="#b65f54", label="ideal y = ŷ")
plt.xlabel("synthetic measured logS (y)")
plt.ylabel("predicted logS (ŷ)")
plt.title("Final held-out Test evaluation")
plt.legend();


## 6. 反作弊关卡：漂亮分数从哪里偷来的？

下面构造一个没有真实信号的任务：`y` 是随机数，500 个 features 也全是随机噪声。按理说模型不该预测得好。

- **错误流程**：先看全体数据（包括 Test 的 `y`），挑出和 `y` 偶然最相关的 feature，再 split。
- **正确流程**：先 split，只用 Train 的 `y` 选 feature，然后原样应用到 Test。

错误流程可能得到更漂亮的 Test 数字，因为 Test 答案已经参与了 feature selection。分数越漂亮，方法学问题越不能被洗白。

In [ ]:
leak_rng = np.random.default_rng(6)
n_demo, p_demo = 80, 5000
X_noise = leak_rng.normal(size=(n_demo, p_demo))
y_noise = leak_rng.normal(size=n_demo)
demo_order = leak_rng.permutation(n_demo)
tr, te = demo_order[:40], demo_order[40:]

def abs_corr_columns(Xm, ym):
    centered_X = Xm - Xm.mean(axis=0)
    centered_y = ym - ym.mean()
    denom = np.sqrt((centered_X ** 2).sum(axis=0) * (centered_y ** 2).sum()) + 1e-12
    return np.abs(centered_X.T @ centered_y / denom)

def fit_one_feature(xtr, ytr, xte):
    A = np.column_stack([xtr, np.ones(len(xtr))])
    w, b = np.linalg.lstsq(A, ytr, rcond=None)[0]
    return w * xte + b

# WRONG: Test targets influence which feature is selected.
illegal_j = int(np.argmax(abs_corr_columns(X_noise, y_noise)))
illegal_pred = fit_one_feature(X_noise[tr, illegal_j], y_noise[tr], X_noise[te, illegal_j])

# RIGHT: Feature selection sees Train only.
legal_j = int(np.argmax(abs_corr_columns(X_noise[tr], y_noise[tr])))
legal_pred = fit_one_feature(X_noise[tr, legal_j], y_noise[tr], X_noise[te, legal_j])

print(f"Illegal select-before-split Test MSE: {mse(y_noise[te], illegal_pred):.3f}")
print(f"Legal split-before-select Test MSE:   {mse(y_noise[te], legal_pred):.3f}")
print("Both datasets contain zero real signal. The illegal score is not evidence of learning.")

## 7. 出口题：把代码翻译成研究设计

不看前文，回答四句：

1. 训练循环中，哪一行产生 prediction？
2. 哪一行更新 parameters？
3. 为什么 scaler 的 `mu / sigma` 只能从 Train 估计？
4. 如果看完 Test 又换 learning rate，原 Test 现在承担了什么角色？

进一步挑战：把 random split 改成按你定义的 `scaffold_id` 或 `year` 分组。先写清未来部署问题，再写 split 代码。

### 可核验资料

- scikit-learn, Cross-validation: https://scikit-learn.org/stable/modules/cross_validation.html
- scikit-learn, Common pitfalls / data leakage: https://scikit-learn.org/stable/common_pitfalls.html
- Dive into Deep Learning, Minibatch SGD: https://d2l.ai/chapter_optimization/minibatch-sgd.html
- Wu et al., MoleculeNet: https://doi.org/10.1039/C7SC02664A
- Delaney, ESOL: https://doi.org/10.1021/ci034243x